# 02 · Message Passing and a GCN Layer — From Scratch

**GNNs for Medicine — Lesson 2 of 5**

In Notebook 01 we saw that `A @ X` sums each node's neighbours. That is the seed of a GNN, but used raw it breaks in two ways. This notebook fixes both and builds a proper **Graph Convolutional Network (GCN) layer** by hand — the exact rule from Kipf & Welling (2017), implemented in ~10 lines of PyTorch.

By the end you'll understand the single equation at the heart of the from-scratch model:

$$ H^{(l+1)} = \sigma\!\left( \hat{A}\, H^{(l)}\, W^{(l)} \right) $$

In [1]:
import numpy as np
import torch
import torch.nn as nn
torch.manual_seed(0)

## 1. Two things wrong with raw `A @ X`

**Problem A — a node forgets itself.** In `A @ X`, node *i*'s new value is built from its *neighbours only*. Its own features vanish. A node should keep a sense of itself.

**Problem B — magnitudes explode.** A node with 20 neighbours gets a sum ~20× larger than a node with 1 neighbour. Stack a few layers and values blow up (or vanish). Let's see it.

In [2]:
# A "star" graph: node 0 is connected to 5 leaves. Node 0 has degree 5, leaves degree 1.
A = np.zeros((6,6))
for leaf in range(1,6):
    A[0,leaf] = A[leaf,0] = 1
X = np.ones((6,1))                      # every node starts at value 1

h1 = A @ X
h2 = A @ (A @ X)
print("After 1 hop:", h1.ravel())       # node 0 -> 5, leaves -> 1
print("After 2 hops:", h2.ravel())      # node 0 -> 5, leaves -> 5 ... magnitudes diverge by degree
print("\nNode 0's value is 5x the leaves' purely because it has more neighbours — not because it's more important.")

After 1 hop: [5. 1. 1. 1. 1. 1.]
After 2 hops: [5. 5. 5. 5. 5. 5.]

Node 0's value is 5x the leaves' purely because it has more neighbours — not because it's more important.


## 2. Fix A — add self-loops

Add an identity matrix: $\tilde{A} = A + I$. Now "aggregate neighbours" includes the node itself, so information is never lost.

## 3. Fix B — symmetric normalization

Scale by node degrees so the aggregation is a **weighted average** instead of a raw sum. The GCN choice is *symmetric* normalization:

$$ \hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2} $$

where $\tilde{D}$ is the diagonal degree matrix of $\tilde{A}$. Intuitively, a message from node *j* to node *i* is divided by $\sqrt{\deg(i)}\cdot\sqrt{\deg(j)}$ — so popular neighbours don't dominate, and magnitudes stay stable across layers.

In [3]:
def normalized_adj(edge_index, n):
    """Build  = D^-1/2 (A + I) D^-1/2  from an edge_index [2, E]."""
    A = torch.zeros((n, n))
    if edge_index.shape[1] > 0:
        A[edge_index[0], edge_index[1]] = 1.0
    A = A + torch.eye(n)                              # self-loops  (A + I)
    deg = A.sum(1)
    dinv = torch.pow(deg, -0.5)
    dinv[torch.isinf(dinv)] = 0.0                     # guard isolated nodes
    D = torch.diag(dinv)
    return D @ A @ D

# same star graph, as edge_index
ei = torch.tensor([[0,1,0,2,0,3,0,4,0,5],
                   [1,0,2,0,3,0,4,0,5,0]])
Ahat = normalized_adj(ei, 6)
Xt = torch.ones((6,1))
print("Normalized adjacency row sums:", Ahat.sum(1).round(decimals=3).tolist())
print("Propagate 3 hops, values stay bounded:")
h = Xt
for k in range(3):
    h = Ahat @ h
    print(f"  hop {k+1}:", h.ravel().round(decimals=3).tolist())

Normalized adjacency row sums: [1.6100000143051147, 0.7889999747276306, 0.7889999747276306, 0.7889999747276306, 0.7889999747276306, 0.7889999747276306]
Propagate 3 hops, values stay bounded:
  hop 1: [1.6100000143051147, 0.7889999747276306, 0.7889999747276306, 0.7889999747276306, 0.7889999747276306, 0.7889999747276306]
  hop 2: [1.406999945640564, 0.859000027179718, 0.859000027179718, 0.859000027179718, 0.859000027179718, 0.859000027179718]
  hop 3: [1.4739999771118164, 0.8360000252723694, 0.8360000252723694, 0.8360000252723694, 0.8360000252723694, 0.8360000252723694]


Notice the values now **stay in a sane range** instead of growing with degree. That stability is what lets us stack several layers.

## 4. A GCN layer

A full layer does three things:

1. **Propagate** — mix in neighbours with $\hat{A}$.
2. **Transform** — multiply by a learnable weight matrix $W$ (this is what the network *learns*).
3. **Activate** — a nonlinearity $\sigma$ (ReLU).

$$ H^{(l+1)} = \sigma\!\left( \hat{A}\, H^{(l)}\, W^{(l)} \right) $$

In code, $H W$ (the learnable transform) is just a `nn.Linear`. That's the whole layer:

In [4]:
class GCNLayer(nn.Module):
    def __init__(self, f_in, f_out):
        super().__init__()
        self.lin = nn.Linear(f_in, f_out)     # this is the learnable W (+ bias)
    def forward(self, Ahat, H):
        return self.lin(Ahat @ H)             #  (Â H) W

# Try it on ethanol (3 atoms). Reuse the featurizer idea from NB01 with a quick inline version.
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
import numpy as np
ATOM=['C','N','O','F','P','S','Cl','Br','I','B','Si','Na','K','other']; HYB=['SP','SP2','SP3','SP3D','SP3D2','other']
def oh(x,c):
    v=[0]*len(c); v[c.index(x) if x in c else len(c)-1]=1; return v
def af(a): return oh(a.GetSymbol(),ATOM)+oh(str(a.GetHybridization()),HYB)+[a.GetDegree()/4,a.GetFormalCharge(),int(a.GetIsAromatic()),a.GetTotalNumHs()/4,int(a.IsInRing())]
def smiles_to_graph(s):
    m=Chem.MolFromSmiles(s); X=np.array([af(a) for a in m.GetAtoms()],dtype=np.float32)
    sr,ds=[],[]
    for b in m.GetBonds():
        i,j=b.GetBeginAtomIdx(),b.GetEndAtomIdx(); sr+=[i,j]; ds+=[j,i]
    ei=np.array([sr,ds]) if sr else np.zeros((2,0),dtype=np.int64)
    return X,ei

X_np, ei_np = smiles_to_graph("CCO")
H = torch.tensor(X_np)
Ahat = normalized_adj(torch.tensor(ei_np), H.shape[0])

layer = GCNLayer(H.shape[1], 8)              # 25 features -> 8
out = layer(Ahat, H)
print("Input : 3 atoms x", H.shape[1], "features")
print("Output: 3 atoms x", out.shape[1], "features (new learned embedding per atom)")
print(out.detach().round(decimals=2))

Input : 3 atoms x 25 features
Output: 3 atoms x 8 features (new learned embedding per atom)
tensor([[ 0.1900, -0.0300,  0.0200, -0.0200,  0.0900,  0.0600, -0.2200,  0.3000],
        [ 0.1400, -0.1400,  0.0200, -0.1000,  0.0100,  0.0700, -0.2200,  0.2700],
        [ 0.0700, -0.1100, -0.0500, -0.1500, -0.0700,  0.0900, -0.1700,  0.2200]])


## 5. Stacking layers = a wider "receptive field"

One layer lets a node see its **immediate** neighbours (1 hop). Stack two layers and each node indirectly sees neighbours-of-neighbours (2 hops), because its neighbours already absorbed *their* neighbours. **k layers → information from k bonds away.**

For molecules this matters: a functional group three bonds away can decide whether a drug crosses the BBB. Let's watch the receptive field grow on caffeine.

In [5]:
X_np, ei_np = smiles_to_graph("CN1C=NC2=C1C(=O)N(C)C(=O)N2C")   # caffeine
H0 = torch.tensor(X_np)
Ahat = normalized_adj(torch.tensor(ei_np), H0.shape[0])

torch.manual_seed(1)
g1, g2, g3 = GCNLayer(H0.shape[1],16), GCNLayer(16,16), GCNLayer(16,16)
relu = nn.ReLU()
h1 = relu(g1(Ahat,H0))
h2 = relu(g2(Ahat,h1))
h3 = relu(g3(Ahat,h2))
for name,h in [("after 1 layer",h1),("after 2 layers",h2),("after 3 layers",h3)]:
    # how similar are all atom embeddings to each other? (mean pairwise cosine)
    hn = torch.nn.functional.normalize(h, dim=1)
    sim = (hn @ hn.T).mean().item()
    print(f"{name}: mean pairwise similarity of atom embeddings = {sim:.3f}")
print("\nSimilarity rises with depth: atoms increasingly 'agree' as their receptive fields overlap.")
print("Pushed too far this becomes OVER-SMOOTHING (all atoms identical) — why GCNs are usually 2-4 layers deep.")

after 1 layer: mean pairwise similarity of atom embeddings = 0.938
after 2 layers: mean pairwise similarity of atom embeddings = 0.980
after 3 layers: mean pairwise similarity of atom embeddings = 0.999

Similarity rises with depth: atoms increasingly 'agree' as their receptive fields overlap.
Pushed too far this becomes OVER-SMOOTHING (all atoms identical) — why GCNs are usually 2-4 layers deep.


## Recap

- Raw neighbour-sums lose the node itself and explode with degree.
- **Self-loops** ($A+I$) + **symmetric normalization** give the stable propagation matrix $\hat{A}$.
- A **GCN layer** is `σ(Â H W)` — propagate, learnable-transform, activate. In code: `Linear(Ahat @ H)`.
- **Depth = hops.** 2–4 layers is the sweet spot; too many causes over-smoothing.

We now have a layer that turns atoms into learned embeddings. But to classify a *whole molecule* we must (a) combine atom embeddings into one molecule vector, and (b) train on thousands of molecules at once.

**Next — Notebook 03:** pooling, mini-batching many graphs, the training loop, and our first real BBBP predictions — all from scratch.